## 📖 Topic 4 — Adding & Modifying Columns

* Why does this matter?

Raw data is never enough. In every real project you'll need to create new columns from existing ones — calculate profit margins, categorize salary bands, extract year from date, clean up text. This is called feature engineering and it's one of the most common things a Data Analyst does daily.

In [2]:
import pandas as pd

data = {
    'Name':   ['Sumed', 'Priya', 'Rahul', 'Neha', 'Arjun'],
    'Dept':   ['Data', 'HR', 'Data', 'Finance', 'Data'],
    'Salary': [60000, 45000, 75000, 55000, 80000],
    'Exp':    [3, 2, 5, 4, 6]
}
df = pd.DataFrame(data)

In [3]:
df

,Name,Dept,Salary,Exp
0,Sumed,Data,60000,3
1,Priya,HR,45000,2
2,Rahul,Data,75000,5
3,Neha,Finance,55000,4
4,Arjun,Data,80000,6


## Adding new column from Arithmatic

In [4]:
# Simplest way: assign directly with df['new_col'] = expression
# Pandas applies the operation row-by-row automatically (vectorized)

# Annual bonus = 10% of salary
df['Bonus'] = df['Salary'] * 0.10



In [5]:
print(df)

    Name     Dept  Salary  Exp   Bonus
0  Sumed     Data   60000    3  6000.0
1  Priya       HR   45000    2  4500.0
2  Rahul     Data   75000    5  7500.0
3   Neha  Finance   55000    4  5500.0
4  Arjun     Data   80000    6  8000.0


In [6]:
# Monthly salary
df['Monthly'] = df['Salary'] / 12

print(df)

    Name     Dept  Salary  Exp   Bonus      Monthly
0  Sumed     Data   60000    3  6000.0  5000.000000
1  Priya       HR   45000    2  4500.0  3750.000000
2  Rahul     Data   75000    5  7500.0  6250.000000
3   Neha  Finance   55000    4  5500.0  4583.333333
4  Arjun     Data   80000    6  8000.0  6666.666667


In [7]:
# Salary per year of experience
df['Sal_per_Exp'] = df['Salary'] / df['Exp']

print(df)

    Name     Dept  Salary  Exp   Bonus      Monthly   Sal_per_Exp
0  Sumed     Data   60000    3  6000.0  5000.000000  20000.000000
1  Priya       HR   45000    2  4500.0  3750.000000  22500.000000
2  Rahul     Data   75000    5  7500.0  6250.000000  15000.000000
3   Neha  Finance   55000    4  5500.0  4583.333333  13750.000000
4  Arjun     Data   80000    6  8000.0  6666.666667  13333.333333


## Modifying an esisting column


* Always use .copy() of original dataframe before modifying

In [8]:
# Same syntax — just use an existing column name
# This overwrites the column in place

# Give everyone a 10% salary raise
df['Salary'] = df['Salary'] * 1.10



In [9]:
# Round to nearest integer
df['Salary'] = df['Salary'].round(0).astype(int)

print(df['Salary'])

0    66000
1    49500
2    82500
3    60500
4    88000
Name: Salary, dtype: int64


## Apply() any function to a column

In [10]:
# apply() → takes a function and applies it to each row/element
# Use when arithmetic alone isn't enough

# Using a named function
def salary_grade(sal):
    if sal >= 75000:
        return 'High'
    elif sal >= 55000:
        return 'Mid'
    else:
        return 'Low'

df['Grade'] = df['Salary'].apply(salary_grade)

# apply() passes each value in 'Salary' into salary_grade()
# one row at a time → returns a result → builds new column
print(df[['Name', 'Salary', 'Grade']])

    Name  Salary Grade
0  Sumed   66000   Mid
1  Priya   49500   Low
2  Rahul   82500  High
3   Neha   60500   Mid
4  Arjun   88000  High


## Lambda with Apply() : one liner function

* axis=0 = apply column-wise (default).
* axis=1 = apply row-wise. 
* You'll use axis=1 whenever your function needs values from multiple columns of the same row.

In [12]:
# lambda → anonymous function, written inline
# Syntax: lambda argument: expression
# Use when the function is simple enough for one line

# Tag senior employees (Exp > 4)
df['Senior'] = df['Exp'].apply(lambda x: 'Yes' if x > 4 else 'No')



In [13]:
# Double the salary (trivial example to show syntax)
df['Double_Sal'] = df['Salary'].apply(lambda x: x * 2)



In [14]:
# apply lambda on MULTIPLE columns using axis=1
# axis=1 means apply row-wise (each row passed as a Series)
df['Summary'] = df.apply(
    lambda row: row['Name'] + ' (' + row['Dept'] + ')', axis=1
)


In [15]:
print(df[['Name', 'Exp', 'Senior', 'Summary']])

    Name  Exp Senior         Summary
0  Sumed    3     No    Sumed (Data)
1  Priya    2     No      Priya (HR)
2  Rahul    5    Yes    Rahul (Data)
3   Neha    4     No  Neha (Finance)
4  Arjun    6    Yes    Arjun (Data)


## MAP() replace values using a Dictionary

In [16]:
# map() → replaces values in a Series using a dictionary or function
# Best for encoding / replacing categorical values
# Works only on Series (single column), not entire DataFrame

# Encode department names to numbers
dept_map = {'Data': 1, 'HR': 2, 'Finance': 3}
df['Dept_Code'] = df['Dept'].map(dept_map)

# Replace Yes/No with 1/0
df['Senior_Flag'] = df['Senior'].map({'Yes': 1, 'No': 0})

print(df[['Name', 'Dept', 'Dept_Code', 'Senior', 'Senior_Flag']])

    Name     Dept  Dept_Code Senior  Senior_Flag
0  Sumed     Data          1     No            0
1  Priya       HR          2     No            0
2  Rahul     Data          1    Yes            1
3   Neha  Finance          3     No            0
4  Arjun     Data          1    Yes            1


## Assign() add column without modifying original

In [17]:
# assign() → returns a NEW DataFrame with added columns
# Original df is NOT changed — safe for pipelines
# Can add multiple columns in one call

df2 = df.assign(
    Tax      = df['Salary'] * 0.20,
    Net_Pay  = df['Salary'] * 0.80
)

print(df2[['Name', 'Salary', 'Tax', 'Net_Pay']])

    Name  Salary      Tax  Net_Pay
0  Sumed   66000  13200.0  52800.0
1  Priya   49500   9900.0  39600.0
2  Rahul   82500  16500.0  66000.0
3   Neha   60500  12100.0  48400.0
4  Arjun   88000  17600.0  70400.0


## Rename column

* ⚠ inplace=True modifies the original DataFrame permanently. inplace=False (default) returns a new DataFrame. Most senior analysts avoid inplace=True — it makes code harder to debug.

In [ ]:
# rename() → rename one or more columns
# Pass a dictionary: {'old_name': 'new_name'}
# inplace=True modifies original. inplace=False returns new df.

df.rename(columns={
    'Exp': 'Experience',
    'Dept': 'Department'
}, inplace=True)

# Rename ALL columns at once
df.columns = ['name', 'dept', 'salary', 'exp']

# Lowercase all column names — very common cleaning step
df.columns = df.columns.str.lower()

## 🔑 What You Must Know Cold


 Direct assignment df['col'] = expression — simplest way, modifies in place, vectorized (no loops)

* *apply(func)* — use when logic is complex (if/elif/else). Applies function to each element of a column
* *lambda x:* — shorthand for simple one-line functions inside apply(). x is each row's value
* *axis=1* — row-wise apply, when your function needs values from multiple columns of the same row
* *map(dict)* — best for encoding/replacing categorical values using a lookup dictionary. 
Series only
* *assign()* — non-destructive, returns new DataFrame. Preferred in clean production code
* *rename(columns={})* — rename specific columns. Avoid inplace=True in professional code